In [9]:
import numpy as np
import pandas as pd
import chromadb
import time

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_reviews.csv")
embeddings = np.load("../data/cleaned/sentence_embeddings.npy")

In [5]:
print(df.shape)
print(embeddings.shape)

(30157, 11)
(30157, 384)


In [3]:
client = chromadb.PersistentClient(path="../data/chromadb")
collection = client.get_or_create_collection(
    name="complaint_embeddings",
    metadata={"hnsw:space":"cosine"} #uses cosine similarity for search
    ) 

In [7]:
print("Collection created: ",collection.name)

Collection created:  complaint_embeddings


In [12]:
# WARNING: Only run this cell once. Collection already persisted to disk.
# Re-running will throw duplicate ID errors.
# To verify: print(collection.count()) — should show 30157

In [12]:
batch_size = 500

for i in range(0,len(df),batch_size):
    batch_df = df.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]

    collection.add(
        ids= [str(i) for i in batch_df.index],
        embeddings=batch_embeddings.tolist(),
        documents=batch_df["text_clean_full"].tolist(),
        metadatas= batch_df[['rating','topic_id','topic_label']].to_dict(orient='records')
    )

print("Total documents added: ",collection.count())

Total documents added:  30157


In [4]:
import re
import emoji
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r'&[a-z]+;|&#\d+;', '', text) 
    text = re.sub(r'<[^>]+>', '', text)
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
def retrieve(query,collection,model,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances']
    )
    return result

In [6]:
results = retrieve("battery not charging",collection,model,top_k=3)
print(results)

{'ids': [['29641', '6404', '10297']], 'embeddings': None, 'documents': [['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'rating': 1.0, 'topic_id': 0, 'topic_label': 'Charging issues'}, {'topic_label': 'Charging issues', 'topic_id': 0, 'rating': 1.0}, {'rating': 1.0, 'topic_id': 0, 'topic_label': 'Charging issues'}]], 'distances': [[0.2036428451538086, 0.2220737338066101, 0.22316217422485352]]}


In [7]:
print("Documents in collection: ",collection.count())

Documents in collection:  30157


In [6]:
def retrieve_filtered(query,collection,model,topic_label,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances'],
        where= {"topic_label":{"$eq": topic_label}}
    )
    return result

In [7]:
results = retrieve_filtered("battery not charging",collection,model,topic_label="Charging issues",top_k=3)
print(results)

{'ids': [['29641', '6404', '10297']], 'embeddings': None, 'documents': [['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'topic_id': 0, 'topic_label': 'Charging issues', 'rating': 1.0}, {'topic_label': 'Charging issues', 'rating': 1.0, 'topic_id': 0}, {'rating': 1.0, 'topic_label': 'Charging issues', 'topic_id': 0}]], 'distances': [[0.2036428451538086, 0.2220737338066101, 0.22316217422485352]]}


In [ ]:
results_unfiltered = retrieve("battery not charging",collection,model,top_k=3)
results_filtered = retrieve_filtered("battery not charging",collection,model,topic_label="Cracked screen/Display",top_k=3)

print("Unfiltered: ")
print(results_unfiltered['documents'])

print("Filtered: ")
print(results_filtered['documents'])

Unfiltered: 
[['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']]
Filtered: 
[['i received it sept 8 2016 it worked for two months and now has stopped working on nov 30 will not charge or turn on', 'after 3 days it stopped working', 'phone stopped working in less than 6 months']]


In [10]:
start = time.time()
results_unfiltered = retrieve("battery not charging",collection,model,top_k=3)
end = time.time()
print(f"Time taken by unfiltered search: {(end-start)*1000} milliseconds")

Time taken by unfiltered search: 41.56208038330078 milliseconds


In [11]:
start = time.time()
results_filtered = retrieve_filtered("battery not charging",collection,model,topic_label="Charging issues",top_k=3)
end = time.time()
print(f"Time taken by filtered search: {(end-start)*1000} milliseconds")

Time taken by filtered search: 101.41539573669434 milliseconds


## Phase 5 — ChromaDB Vector Store

### Setup
- **Client:** PersistentClient saved to `data/chromadb/` — persists across sessions
- **Collection:** `complaint_embeddings` with cosine similarity (HNSW index)
- **Documents loaded:** 30,157 complaints in batches of 500
- **Stored per document:** embedding (384-dim), text (`text_clean_full`),
  metadata (`rating`, `topic_id`, `topic_label`)

### Functions Built

**`retrieve(query, collection, model, top_k=3)`**
Semantic search across full corpus. Preprocesses query, embeds it, returns
top-k similar complaints with documents, metadata, and cosine distances.

**`retrieve_filtered(query, collection, model, topic_label, top_k=3)`**
Same as retrieve() but restricted to a specific topic using ChromaDB's
`$eq` metadata filter — ensures retrieved cases match the complaint category.

### Latency Results
| Function | Latency |
|----------|---------|
| `retrieve()` | ~42ms |
| `retrieve_filtered()` | ~101ms |

### Why Filtered Search is Slower
ChromaDB applies metadata filters **post-retrieval** — it searches the full
index first, then filters by metadata. More candidates are evaluated to find
enough matches within the target topic.

**Production fix:** Partition collection by topic — one collection per topic
so vector search is constrained upfront, not filtered after.

### Distance Interpretation
ChromaDB returns cosine **distance** (0=identical, 1=unrelated) — inverse of
cosine similarity. Best match for "battery not charging" scored 0.20 distance
= 80% similarity. Threshold for Confidence Gate (Phase 7): distance > 0.4
(below 60% similarity) → fallback to direct generation, skip RAG.

### Key Verification
- Unfiltered "battery not charging" → 3 Charging Issues complaints ✅
- Filtered to "Cracked screen/Display" → returns screen-topic complaints
  even for a charging query — demonstrates filter override working correctly ✅